# 欠損値の補完
欠損値を、列の性質に合った値で埋めます(補完)。カテゴリの変数は最頻値、数値の変数は中央値、年齢はグループごとの中央値で埋め方を工夫します。欠損の多い客室番号は「記録があるかどうか」の特徴量に変えます。

## 学習目標
- `fillna` で欠損値を埋められる
- カテゴリの変数は最頻値、数値の変数は中央値で補完する理由を説明できる
- グループごとの中央値(`groupby` + `map`)で、より適した値を補完できる
- 欠損の多い列を「記録の有無」を表す特徴量に変えられる

## 補完の方針

| 列 | 欠損(学習用) | 補完の方法 |
|---|---|---|
| `Embarked` | 2 人 | 最も多い港(最頻値)で埋める |
| `Fare` | 1 人 | 中央値で埋める(前の単元の演習) |
| `Age` | 173 人 | グループ(敬称)ごとの中央値で埋める |
| `Cabin` | 695 人 | 埋めずに、「記録があるか」の列(`HasCabin`)に変える |

## カテゴリの変数:最頻値で埋める

港のようなカテゴリの変数は、平均や中央値が計算できないので、**最も多い値(最頻値)** で埋めます。`mode()` は最頻値を返します(同数の値が複数あると複数返すので、`[0]` で先頭を取り出します)。

In [1]:
import pandas as pd

# 学習用・予測用のデータを読み込む
train = pd.read_csv("train.csv")
test = pd.read_csv("test.csv")

# 学習用のデータから、乗船した港の最頻値を求める
embarked_mode = train["Embarked"].mode()[0]
print("最頻値:", embarked_mode)

# 欠損値を最頻値で埋め、埋めた後の欠損値の数を確認する
train["Embarked"] = train["Embarked"].fillna(embarked_mode)
print("欠損値の数:", train["Embarked"].isna().sum())

最頻値: S
欠損値の数: 0


欠損していた 2 人の港は、最も多い S(Southampton)で埋まりました。欠損がごく少ないので、どの値で埋めても全体への影響はわずかです。

## 年齢:グループごとの中央値で埋める

年齢は 173 人(約 2 割)が欠損しているので、埋め方の影響が大きくなります。全員を同じ中央値(29 歳)で埋めると、子どもも 29 歳として扱われてしまいます。

そこで、年齢と関係のある列で **グループに分け、グループごとの中央値** で埋めます。例として、客室の等級ごとの中央値を求めてみます。

In [2]:
# 学習用のデータから、等級ごとの年齢の中央値を求める
age_by_class = train.groupby("Pclass")["Age"].median()
print(age_by_class)

Pclass
1    39.0
2    28.0
3    24.0
Name: Age, dtype: float64


1 等は 39 歳、3 等は 24 歳と、等級によって中央値が大きく違います。各乗客の等級に対応する中央値は、`map` で取り出せます。`map` に「等級 → 中央値」の対応表(Series)を渡すと、各行の等級を中央値に置き換えた列が得られます。

In [3]:
# 各乗客の等級を、その等級の年齢の中央値に置き換える
fill_values = train["Pclass"].map(age_by_class)

# 年齢の欠損値だけを、対応する中央値で埋める
age_filled = train["Age"].fillna(fill_values)

# 埋める前と後の、欠損値の数と平均を比べる
print("欠損値の数:", train["Age"].isna().sum(), "→", age_filled.isna().sum())
print("平均:", round(train["Age"].mean(), 2), "→", round(age_filled.mean(), 2))

欠損値の数: 173 → 0
平均: 30.48 → 29.62


欠損値はなくなり、平均の変化も小さく抑えられています。この講座では、等級よりも年齢との関係が強い **敬称**(Master は男の子、Miss は女の子を含む未婚女性など)のグループで埋めることにします(演習で行います)。

## 客室番号:記録の有無に変える

客室番号は約 8 割が欠損しているので、埋めるのではなく、**記録があれば 1、なければ 0** の列に変えます。前の講座で、記録の有無によって生存率が大きく違う(64% と 30%)ことを確かめました。

In [4]:
# notna() で「記録がある」を True にし、astype(int) で True → 1、False → 0 に変える
train["HasCabin"] = train["Cabin"].notna().astype(int)

# 新しい列の値ごとの人数を確認する
print(train["HasCabin"].value_counts())

HasCabin
0    695
1    196
Name: count, dtype: int64


## 演習

学習用のデータで、名前から敬称を取り出して `Title` 列を作り、**敬称ごとの年齢の中央値** で年齢の欠損値を埋めてください。敬称ごとの中央値(人数の多い順に上位 4 つ)と、埋めた後の欠損値の数を表示してください。

In [ ]:
# @starter
import pandas as pd

# 学習用のデータを読み込む
train = pd.read_csv("train.csv")

# 名前から敬称を取り出す(前の講座で使った正規表現)
train["Title"] = train["Name"].str.extract(r",\s*([^.]+)\.")[0]

# 敬称ごとの年齢の中央値を求める(____ に、グループ分けに使う列の名前を入れる)
age_by_title = train.groupby("____")["Age"].median()

# 人数の多い敬称の上位 4 つについて、中央値を表示する
top4 = train["Title"].value_counts().index[:4]
print(age_by_title[top4])

# 各乗客の敬称を中央値に置き換え、年齢の欠損値を埋める(____ に、置き換えに使うメソッドの名前を入れる)
train["Age"] = train["Age"].fillna(train["Title"].____(age_by_title))

# 埋めた後の欠損値の数を表示する
print(train["Age"].isna().sum())

In [5]:
# @solution
# @title: 敬称ごとの中央値で年齢を埋める
# @hint: グループごとの中央値は groupby("Title")["Age"].median()、各行の敬称を中央値に置き換えるには map を使います。
# @explanation: Master(男の子)の中央値は 5.5 歳、Miss は 21 歳、Mr は 29 歳、Mrs は 36 歳です。敬称で分けると、子どもの年齢を大人の中央値で埋めてしまうことを避けられます。
import pandas as pd

# 学習用のデータを読み込む
train = pd.read_csv("train.csv")

# 名前から敬称を取り出す(前の講座で使った正規表現)
train["Title"] = train["Name"].str.extract(r",\s*([^.]+)\.")[0]

# 敬称ごとの年齢の中央値を求める
age_by_title = train.groupby("Title")["Age"].median()

# 人数の多い敬称の上位 4 つについて、中央値を表示する
top4 = train["Title"].value_counts().index[:4]
print(age_by_title[top4])

# 各乗客の敬称を中央値に置き換え(map)、年齢の欠損値を埋める
train["Age"] = train["Age"].fillna(train["Title"].map(age_by_title))

# 埋めた後の欠損値の数を表示する
print(train["Age"].isna().sum())

Title
Mr        29.0
Miss      21.0
Mrs       36.0
Master     5.5
Name: Age, dtype: float64
0


予測用のデータ `test.csv` に、客室番号の記録の有無を表す `HasCabin` 列(記録があれば 1、なければ 0)を作り、値ごとの人数を表示してください。

In [ ]:
# @starter
import pandas as pd

# 予測用のデータを読み込む
test = pd.read_csv("test.csv")

# 記録があれば 1、なければ 0 の列を作る
# (____ に、欠損でないかどうかを判定するメソッドの名前と、True/False を 1/0 に変える処理を書く)
test["HasCabin"] = test["Cabin"].____

# 値ごとの人数を表示する
print(test["HasCabin"].value_counts())

In [6]:
# @solution
# @title: 予測用のデータに HasCabin を作る
# @hint: notna() で記録の有無を True / False にし、astype(int) で 1 / 0 に変えます。
# @explanation: 予測用のデータでは 99 人に客室番号の記録があります。学習用のデータと同じ方法で列を作ることで、モデルは同じ意味の特徴量として扱えます。
import pandas as pd

# 予測用のデータを読み込む
test = pd.read_csv("test.csv")

# notna() で「記録がある」を True にし、astype(int) で True → 1、False → 0 に変える
test["HasCabin"] = test["Cabin"].notna().astype(int)

# 値ごとの人数を表示する
print(test["HasCabin"].value_counts())

HasCabin
0    319
1     99
Name: count, dtype: int64


<!-- @question
type: choice
answer: 子どもの年齢も大人に近い値で埋められ、年齢と生存の関係(子どもは生存しやすい)が弱まってしまうおそれがある。
title: 全員を同じ中央値で埋める問題点
hint: 年齢が欠損している乗客の中には、子どもも含まれています。前の講座で、年齢と生存にはどのような関係がありましたか。
explanation: 全員を 29 歳で埋めると、年齢が欠損していた子どもも 29 歳として扱われます。前の講座で確かめた「子どもは生存しやすい」という関係が、補完によって弱まってしまうおそれがあります。敬称(Master など)のように年齢と関係のある列でグループに分けて埋めると、この問題を小さくできます。
-->
年齢の欠損値を、敬称などで分けずに全員同じ中央値(29 歳)で埋めた場合の問題点として最も適切なものを選べ。

- 埋めた後も欠損値が残ってしまう。
- 子どもの年齢も大人に近い値で埋められ、年齢と生存の関係(子どもは生存しやすい)が弱まってしまうおそれがある。
- 中央値は外れ値の影響を受けやすいので、平均で埋める必要がある。
- 年齢の列が文字列になってしまい、モデルに入れられなくなる。

## まとめ

- カテゴリの変数は最頻値(`mode()[0]`)、数値の変数は中央値で `fillna` する。値は学習用のデータから求める。
- 欠損の多い・影響の大きい列は、関係のある列のグループごとの中央値(`groupby` + `map`)で埋める。年齢は敬称ごとに埋める。
- 欠損の多い客室番号は、記録の有無(`HasCabin`)の特徴量に変える。